In [11]:
import cv2
import mediapipe as mp
import numpy as np
import pyttsx3
import threading
import time
import os
import json
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pickle

print("✅ All libraries loaded!")

✅ All libraries loaded!


In [12]:
import subprocess
subprocess.run(["pip", "install", "scikit-learn"], check=True)
print("✅ scikit-learn ready!")

✅ scikit-learn ready!


In [13]:
last_spoken = {"text": "", "time": 0}
COOLDOWN = 1.5

def speak(text):
    now = time.time()
    if text == last_spoken["text"] and (now - last_spoken["time"]) < COOLDOWN:
        return
    last_spoken["text"] = text
    last_spoken["time"] = now

    def _run():
        engine = pyttsx3.init()
        engine.setProperty('rate', 140)
        engine.setProperty('volume', 1.0)
        engine.say(text)
        engine.runAndWait()

    threading.Thread(target=_run, daemon=True).start()

speak("System ready.")
time.sleep(2)
print("✅ TTS ready!")

✅ TTS ready!


In [14]:
mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_styles  = mp.solutions.drawing_styles

# ASL Alphabet + common ASL words
ASL_LABELS = [
    # Letters
    "A", "B", "C", "D", "E", "F", "G", "H", "I", "J",
    "K", "L", "M", "N", "O", "P", "Q", "R", "S", "T",
    "U", "V", "W", "X", "Y", "Z",
    # Common words
    "Hello", "Thank You", "Yes", "No", "Please",
    "Sorry", "Help", "Stop", "I Love You", "Good"
]

COLORS = {
    # Letters — blue tones
    **{letter: (200, 100, 0) for letter in "ABCDEFGHIJKLMNOPQRSTUVWXYZ"},
    # Words — distinct colors
    "Hello":     (0,   220, 120),
    "Thank You": (0,   200, 255),
    "Yes":       (0,   255, 100),
    "No":        (0,   60,  220),
    "Please":    (200, 0,   255),
    "Sorry":     (255, 100, 0  ),
    "Help":      (255, 200, 0  ),
    "Stop":      (0,   0,   220),
    "I Love You":(255, 20,  147),
    "Good":      (100, 255, 100),
    "Unknown":   (80,  80,  80 ),
    "No Hand":   (60,  60,  60 ),
}

def extract_features(landmarks):
    """
    Extract normalized 63-feature vector from 21 hand landmarks.
    Normalized to wrist position and hand scale for accuracy.
    """
    wrist  = landmarks[0]
    coords = []
    for lm in landmarks:
        coords.append(lm.x - wrist.x)
        coords.append(lm.y - wrist.y)
        coords.append(lm.z - wrist.z)

    # Normalize by hand scale
    scale = np.sqrt(
        (landmarks[9].x - wrist.x) ** 2 +
        (landmarks[9].y - wrist.y) ** 2 +
        (landmarks[9].z - wrist.z) ** 2
    )
    if scale > 0:
        coords = [c / scale for c in coords]

    return np.array(coords)

def draw_hud(frame, label, word_buffer, sentence, confidence=None):
    h, w = frame.shape[:2]
    color = COLORS.get(label, (150, 150, 150))

    # Top bar
    cv2.rectangle(frame, (0, 0), (w, 110), (10, 10, 10), -1)

    # Current sign
    cv2.putText(frame, f"Sign: {label}", (15, 38),
                cv2.FONT_HERSHEY_DUPLEX, 1.1, color, 2, cv2.LINE_AA)

    # Confidence
    if confidence is not None:
        conf_color = (0, 255, 0) if confidence > 85 else (0, 165, 255) if confidence > 65 else (0, 0, 255)
        cv2.putText(frame, f"Confidence: {confidence:.1f}%", (15, 68),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, conf_color, 1, cv2.LINE_AA)

    # Word being built
    cv2.putText(frame, f"Word: {word_buffer}", (15, 95),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 100), 1, cv2.LINE_AA)

    # Sentence at bottom
    cv2.rectangle(frame, (0, h - 50), (w, h), (10, 10, 10), -1)
    cv2.putText(frame, f"Sentence: {sentence[-60:]}", (15, h - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1, cv2.LINE_AA)

    # Hint
    cv2.putText(frame, "SPACE=confirm word  ENTER=speak  BACK=delete  Q=quit",
                (15, h - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (80, 80, 80), 1, cv2.LINE_AA)

    # Border
    cv2.rectangle(frame, (0, 0), (w - 1, h - 1), color, 3)

print("✅ ASL labels and feature extractor ready!")
print(f"📋 Total signs to train: {len(ASL_LABELS)}")

✅ ASL labels and feature extractor ready!
📋 Total signs to train: 36


In [15]:
# ================================================================
#  COLLECT ASL TRAINING DATA
#  One gesture at a time.
#  Hold the sign → Press SPACE to record 150 samples → next sign
#  Press Q to skip, ESC to stop early and save progress
# ================================================================

SAMPLES_PER_SIGN = 150
DATA_FILE = "asl_data.pkl"

all_features = []
all_labels   = []

# Load existing data
if os.path.exists(DATA_FILE):
    with open(DATA_FILE, "rb") as f:
        saved = pickle.load(f)
        all_features = saved["features"]
        all_labels   = saved["labels"]
    already_done = set(all_labels)
    print(f"📂 Loaded existing: {len(all_labels)} samples")
    print(f"✅ Already collected: {already_done}")
else:
    already_done = set()

# Only collect signs not yet done
to_collect = [s for s in ASL_LABELS if s not in already_done]
print(f"\n📋 Signs remaining: {len(to_collect)}")
print("Press SPACE to start recording each sign. Q=skip. ESC=save & stop.")

cap = cv2.VideoCapture(0)
stop_all = False

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.75,
    min_tracking_confidence=0.65
) as hands:

    for sign in to_collect:
        if stop_all:
            break

        collected  = 0
        collecting = False
        print(f"\n👉 Show sign: [{sign}]  —  Press SPACE to record")

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame   = cv2.flip(frame, 1)
            rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            results = hands.process(rgb)
            rgb.flags.writeable = True

            if results.multi_hand_landmarks:
                for hand_lm in results.multi_hand_landmarks:
                    mp_drawing.draw_landmarks(
                        frame, hand_lm,
                        mp_hands.HAND_CONNECTIONS,
                        mp_styles.get_default_hand_landmarks_style(),
                        mp_styles.get_default_hand_connections_style()
                    )
                    if collecting:
                        feat = extract_features(hand_lm.landmark)
                        all_features.append(feat)
                        all_labels.append(sign)
                        collected += 1

            # HUD
            status_color = (0, 255, 0) if collecting else (0, 165, 255)
            status_text  = f"Recording {collected}/{SAMPLES_PER_SIGN}" if collecting else "SPACE to record"
            cv2.rectangle(frame, (0, 0), (frame.shape[1], 90), (10, 10, 10), -1)
            cv2.putText(frame, f"ASL Sign: {sign}", (15, 38),
                        cv2.FONT_HERSHEY_DUPLEX, 1.1, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, status_text, (15, 72),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2, cv2.LINE_AA)
            cv2.putText(frame, "Q=skip  ESC=save & stop", (15, frame.shape[0] - 15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (100, 100, 100), 1, cv2.LINE_AA)

            cv2.imshow("ASL Data Collection", frame)

            key = cv2.waitKey(1) & 0xFF
            if key == ord(' '):
                collecting = True
            elif key == ord('q'):
                print(f"  ⏭ Skipped: {sign}")
                break
            elif key == 27:
                print("  💾 Saving and stopping...")
                stop_all = True
                break

            if collected >= SAMPLES_PER_SIGN:
                print(f"  ✅ Done: {sign} ({collected} samples)")
                break

# Save
with open(DATA_FILE, "wb") as f:
    pickle.dump({"features": all_features, "labels": all_labels}, f)

cap.release()
cv2.destroyAllWindows()
print(f"\n✅ Total samples saved: {len(all_labels)}")
print("Breakdown:")
for s in ASL_LABELS:
    c = all_labels.count(s)
    if c > 0:
        print(f"  {s}: {c} samples")

📂 Loaded existing: 4650 samples
✅ Already collected: {'E', 'I', 'U', 'T', 'S', 'D', 'Q', 'Yes', 'W', 'L', 'V', 'H', 'M', 'Thank You', 'C', 'X', 'Y', 'Z', 'P', 'N', 'Hello', 'K', 'No', 'Please', 'F', 'B', 'A', 'O', 'R', 'J', 'G'}

📋 Signs remaining: 5
Press SPACE to start recording each sign. Q=skip. ESC=save & stop.

👉 Show sign: [Sorry]  —  Press SPACE to record
  ⏭ Skipped: Sorry

👉 Show sign: [Help]  —  Press SPACE to record


I0000 00:00:1775350348.704839       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4


  ⏭ Skipped: Help

👉 Show sign: [Stop]  —  Press SPACE to record
  ⏭ Skipped: Stop

👉 Show sign: [I Love You]  —  Press SPACE to record
  ⏭ Skipped: I Love You

👉 Show sign: [Good]  —  Press SPACE to record
  ✅ Done: Good (150 samples)

✅ Total samples saved: 4800
Breakdown:
  A: 150 samples
  B: 150 samples
  C: 150 samples
  D: 150 samples
  E: 150 samples
  F: 150 samples
  G: 150 samples
  H: 150 samples
  I: 150 samples
  J: 150 samples
  K: 150 samples
  L: 150 samples
  M: 150 samples
  N: 150 samples
  O: 150 samples
  P: 150 samples
  Q: 150 samples
  R: 150 samples
  S: 150 samples
  T: 150 samples
  U: 150 samples
  V: 150 samples
  W: 150 samples
  X: 150 samples
  Y: 150 samples
  Z: 150 samples
  Hello: 150 samples
  Thank You: 150 samples
  Yes: 150 samples
  No: 150 samples
  Please: 150 samples
  Good: 150 samples


In [16]:
# ================================================================
#  TRAIN NEURAL NETWORK ON ASL DATA
# ================================================================

MODEL_FILE = "asl_model.pkl"

with open(DATA_FILE, "rb") as f:
    saved = pickle.load(f)

X = np.array(saved["features"])
y = np.array(saved["labels"])

print(f"📊 Training on {len(X)} samples, {len(set(y))} signs")

le        = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

model = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    max_iter=1000,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.15,
    learning_rate_init=0.001,
    verbose=False
)

print("🧠 Training... (this may take a minute)")
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test) * 100
print(f"\n✅ Training complete!")
print(f"🎯 Test accuracy: {accuracy:.2f}%")

with open(MODEL_FILE, "wb") as f:
    pickle.dump({"model": model, "encoder": le}, f)

print(f"💾 Model saved to {MODEL_FILE}")

📊 Training on 4800 samples, 32 signs
🧠 Training... (this may take a minute)

✅ Training complete!
🎯 Test accuracy: 99.79%
💾 Model saved to asl_model.pkl


In [17]:
# ================================================================
#  REAL-TIME ASL RECOGNITION — WORD BY WORD
#
#  HOW TO USE:
#  - Show a letter or word sign
#  - Press SPACE to confirm and add to word buffer
#  - Press ENTER to speak the full sentence
#  - Press BACKSPACE to delete last letter
#  - Press C to clear everything
#  - Press Q to quit
# ================================================================

with open(MODEL_FILE, "rb") as f:
    saved  = pickle.load(f)
    model  = saved["model"]
    le     = saved["encoder"]

CONFIDENCE_THRESHOLD = 70.0
SMOOTH_FRAMES        = 7
AUTO_CONFIRM_PAUSE   = 2.0  # seconds of same sign = auto confirm letter

recent_preds   = []
word_buffer    = ""
sentence       = ""
last_sign      = ""
last_sign_time = time.time()
last_confirmed = ""
last_confirm_time = 0

cap = cv2.VideoCapture(0)
print("✅ Camera open. Start signing!")

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.75,
    min_tracking_confidence=0.65
) as hands:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame   = cv2.flip(frame, 1)
        rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        results = hands.process(rgb)
        rgb.flags.writeable = True

        label      = "No Hand"
        confidence = None

        if results.multi_hand_landmarks:
            for hand_lm in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    frame, hand_lm,
                    mp_hands.HAND_CONNECTIONS,
                    mp_styles.get_default_hand_landmarks_style(),
                    mp_styles.get_default_hand_connections_style()
                )

                features = extract_features(hand_lm.landmark).reshape(1, -1)
                proba    = model.predict_proba(features)[0]
                top_idx  = np.argmax(proba)
                confidence = proba[top_idx] * 100

                if confidence >= CONFIDENCE_THRESHOLD:
                    predicted = le.inverse_transform([top_idx])[0]
                    recent_preds.append(predicted)
                    if len(recent_preds) > SMOOTH_FRAMES:
                        recent_preds.pop(0)
                    label = max(set(recent_preds), key=recent_preds.count)
                else:
                    label = "Unknown"
                    recent_preds.clear()

                # Auto-confirm letter after holding still
                now = time.time()
                if label == last_sign and label not in ("Unknown", "No Hand"):
                    if (now - last_sign_time) >= AUTO_CONFIRM_PAUSE:
                        if label != last_confirmed or (now - last_confirm_time) > 2.0:
                            # Single letters go to word buffer, words go to sentence
                            if len(label) == 1:
                                word_buffer += label
                            else:
                                if word_buffer:
                                    sentence += word_buffer + " "
                                    word_buffer = ""
                                sentence += label + " "
                            last_confirmed    = label
                            last_confirm_time = now
                            last_sign_time    = now  # reset timer
                else:
                    last_sign      = label
                    last_sign_time = time.time()

        draw_hud(frame, label, word_buffer, sentence, confidence)
        cv2.imshow("ASL to Speech — Real Time", frame)

        key = cv2.waitKey(1) & 0xFF

        # SPACE — manually confirm current sign
        if key == ord(' '):
            if label not in ("Unknown", "No Hand"):
                if len(label) == 1:
                    word_buffer += label
                else:
                    if word_buffer:
                        sentence += word_buffer + " "
                        word_buffer = ""
                    sentence += label + " "

        # ENTER — speak full sentence
        elif key == 13:
            if word_buffer:
                sentence += word_buffer + " "
                word_buffer = ""
            if sentence.strip():
                speak(sentence.strip())
                print(f"🔊 Speaking: {sentence.strip()}")
            sentence = ""

        # BACKSPACE — delete last letter
        elif key == 8:
            if word_buffer:
                word_buffer = word_buffer[:-1]
            elif sentence:
                sentence = sentence.rstrip()
                sentence = sentence[:sentence.rfind(' ') + 1] if ' ' in sentence else ""

        # W — confirm word (add space)
        elif key == ord('w'):
            if word_buffer:
                sentence   += word_buffer + " "
                word_buffer = ""

        # C — clear all
        elif key == ord('c'):
            word_buffer = ""
            sentence    = ""

        # Q — quit
        elif key == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
print("✅ Session ended.")

✅ Camera open. Start signing!


I0000 00:00:1775350364.179267       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4


🔊 Speaking: Good Good Good Thank You Good Good
🔊 Speaking: Good
🔊 Speaking: Good
🔊 Speaking: Good Good
🔊 Speaking: Good
🔊 Speaking: R
✅ Session ended.
